# Pre-processing

In [1]:
import json

import pandas as pd
from pathlib import Path

RESULTS_DIR = Path('/home/oliver1024/Documents/paper_writing/code_base/gift_eval/gift-eval/results')

EXCLUDE_FOLDERS = {'LSGT-PFN', 'LSGT-PFN-v14', 'LSGT-PFN-torch'}

TARGET_DATASETS = ['m4_yearly']

TARGET_COLUMNS = [
    'eval_metrics/MASE[0.5]',
    'eval_metrics/sMAPE[0.5]',
    'eval_metrics/mean_weighted_sum_quantile_loss'
]

# Metadata pulled from each model folder's config.json
META_COLUMNS = ['model_type', 'testdata_leakage']

POSSIBLE_FILENAMES = ['all_results.csv', 'all-results.csv']

all_dfs = []

for folder in sorted(RESULTS_DIR.iterdir()):
    if not folder.is_dir():
        continue

    if folder.name in EXCLUDE_FOLDERS:
        print(f'Skipping: {folder.name}')
        continue

    # Find which CSV exists
    csv_path = None
    for fname in POSSIBLE_FILENAMES:
        candidate = folder / fname
        if candidate.exists():
            csv_path = candidate
            break

    if csv_path is None:
        print(f'No matching CSV in: {folder.name}')
        continue

    df = pd.read_csv(csv_path)

    # Identify dataset column
    dataset_col = None
    for candidate in ['dataset', 'item_id', 'name', 'dataset_name']:
        if candidate in df.columns:
            dataset_col = candidate
            break

    if dataset_col is None:
        print(f'Could not find dataset column in: {folder.name}. Columns: {df.columns.tolist()}')
        continue

    # Filter rows
    filtered = df[
        df[dataset_col]
        .astype(str)
        .str.split('/')
        .str[0]
        .isin(TARGET_DATASETS)
    ].copy()

    if filtered.empty:
        print(f'No matching rows in: {folder.name}')
        continue

    # Keep dataset col + target metric columns
    cols_to_keep = [dataset_col] + [c for c in TARGET_COLUMNS if c in df.columns]
    filtered = filtered[cols_to_keep].copy()
    filtered['model'] = folder.name

    # Attach metadata from config.json (missing file / missing keys -> NA)
    config_path = folder / 'config.json'
    config = {}
    if config_path.exists():
        try:
            with open(config_path) as fh:
                config = json.load(fh)
        except (json.JSONDecodeError, OSError) as e:
            print(f'Could not parse config.json in {folder.name}: {e}')
    else:
        print(f'No config.json in: {folder.name}')

    for col in META_COLUMNS:
        filtered[col] = config.get(col, pd.NA)

    all_dfs.append(filtered)
    print(f'Loaded {len(filtered)} rows from: {folder.name}')

if not all_dfs:
    print("No data collected.")
else:
    combined = pd.concat(all_dfs, ignore_index=True)
    print(combined.shape)
    print(combined.head())

Loaded 1 rows from: CHARM
Loaded 1 rows from: CastStar
Loaded 1 rows from: Chronos_small
Loaded 1 rows from: CleanTS-65M
Loaded 1 rows from: Cobra-Agent
Loaded 1 rows from: Credence
Loaded 1 rows from: DLinear
Loaded 1 rows from: DeOSAlphaTimeGPTPredictor-2025
Loaded 1 rows from: FFM
Loaded 1 rows from: FLAIR
Loaded 1 rows from: Falcon-2.0
Loaded 1 rows from: Falcon-Agent
Loaded 1 rows from: Falcon-X
Loaded 1 rows from: FlowState-9.1M
Loaded 1 rows from: FlowState-r1.1
No config.json in: GRAIN
Loaded 1 rows from: GRAIN
Loaded 1 rows from: Granite-FlowState-r1.1
Loaded 1 rows from: Granite-PatchTST-FM-r1
Loaded 1 rows from: HistRoute-CV
Loaded 1 rows from: Kairos_10m
Loaded 1 rows from: Kairos_23m
Loaded 1 rows from: Kairos_50m
Skipping: LSGT-PFN
Skipping: LSGT-PFN-v14
Loaded 1 rows from: Lag-Llama
Loaded 1 rows from: Lingjiang
Loaded 1 rows from: LongSeer-v1.0
Loaded 1 rows from: Migas-1.0
Loaded 1 rows from: Moirai2
Loaded 1 rows from: MoiraiAgent
Loaded 1 rows from: MoiraiAgent-leaki

In [2]:
# Split by frequency from the dataset column
combined['freq'] = combined['dataset'].str.split('/').str[1]

yearly = combined[combined['freq'] == 'A'].drop(columns='freq').reset_index(drop=True)

yearly.to_csv('results_yearly.csv', index=False)

print(f'Yearly: {len(yearly)} rows')

Yearly: 109 rows


In [3]:
yearly_sorted = yearly.sort_values('eval_metrics/MASE[0.5]').reset_index(drop=True)


yearly_sorted.to_csv('results_yearly.csv', index=False)


print(yearly_sorted)


               dataset  eval_metrics/MASE[0.5]  eval_metrics/sMAPE[0.5]  \
0    m4_yearly/A/short                2.398107                 0.158700   
1    m4_yearly/A/short                2.535312                 0.092667   
2    m4_yearly/A/short                2.537595                 0.109875   
3    m4_yearly/A/short                2.898559                 0.130911   
4    m4_yearly/A/short                2.957246                 0.135139   
..                 ...                     ...                      ...   
104  m4_yearly/A/short                6.026356                 0.264514   
105  m4_yearly/A/short                6.379586                 0.206064   
106  m4_yearly/A/short                7.593336                 0.331697   
107  m4_yearly/A/short                9.565384                 0.325159   
108  m4_yearly/A/short               31.400000                 0.706000   

     eval_metrics/mean_weighted_sum_quantile_loss  \
0                                        0.126

In [4]:
# Load all rlgt csvs
rlgt_yearly_df = pd.read_csv('/home/oliver1024/Documents/paper_writing/code_base/gift_eval/gift-eval/results/GRAIN/all_results.csv')


# Yearly
rlgt_yearly_avg = rlgt_yearly_df.groupby('model')[['eval_metrics/MASE[0.5]', 'eval_metrics/sMAPE[0.5]', 'eval_metrics/mean_weighted_sum_quantile_loss']].mean().reset_index()

# LSGT-PFN folders have no config.json, so declare their metadata here
LSGT_META = {'model_type': 'zero-shot', 'testdata_leakage': 'No'}
for col in META_COLUMNS:
    rlgt_yearly_avg[col] = LSGT_META.get(col, pd.NA)

yearly_final = pd.concat([yearly_sorted, rlgt_yearly_avg], ignore_index=True)
yearly_final = yearly_final.sort_values('eval_metrics/MASE[0.5]').reset_index(drop=True)

# Rank by MASE, drop dataset, reorder columns
yearly_final['rank'] = yearly_final['eval_metrics/MASE[0.5]'].rank(method='min').astype(int)
yearly_final = yearly_final[['rank', 'model'] + META_COLUMNS + TARGET_COLUMNS]

yearly_final.to_csv('results_yearly.csv', index=False)


print('Yearly:'); print(yearly_final)

Yearly:
     rank                             model     model_type testdata_leakage  \
0       1                    tempo_ensemble     fine-tuned              Yes   
1       2  TurkForecast-FM-Chronos2-LoRA-v1     fine-tuned               No   
2       3                  timesfm_2_0_500m     pretrained              Yes   
3       4                    LSGT-PFN-torch      zero-shot               No   
4       4                             GRAIN            NaN              NaN   
..    ...                               ...            ...              ...   
105   106                               FFM  deep-learning               No   
106   107               STRIDE (+Chronos-2)     pretrained               No   
107   108                             CHARM      zero-shot               No   
108   109                         Lag-Llama     pretrained              Yes   
109   110                       crossformer  deep-learning               No   

     eval_metrics/MASE[0.5]  eval_metrics/s

In [5]:
# One CSV per metric, each ranked by that metric (lower is better)
OUTPUTS = {
    'eval_metrics/MASE[0.5]': 'results_yearly_mase.csv',
    'eval_metrics/sMAPE[0.5]': 'results_yearly_smape.csv',
    'eval_metrics/mean_weighted_sum_quantile_loss': 'results_yearly_quantile.csv',
}

for metric, out_path in OUTPUTS.items():
    df = yearly_final[['model'] + META_COLUMNS + [metric]].dropna(subset=[metric])
    df = df.sort_values(metric).reset_index(drop=True)
    df['rank'] = df[metric].rank(method='min').astype(int)
    df = df[['rank', 'model'] + META_COLUMNS + [metric]]
    df.to_csv(out_path, index=False)
    print(f'{out_path}: {len(df)} rows')
    print(df.head(), '\n')

results_yearly_mase.csv: 110 rows
   rank                             model  model_type testdata_leakage  \
0     1                    tempo_ensemble  fine-tuned              Yes   
1     2  TurkForecast-FM-Chronos2-LoRA-v1  fine-tuned               No   
2     3                  timesfm_2_0_500m  pretrained              Yes   
3     4                    LSGT-PFN-torch   zero-shot               No   
4     4                             GRAIN         NaN              NaN   

   eval_metrics/MASE[0.5]  
0                2.398107  
1                2.535312  
2                2.537595  
3                2.898559  
4                2.898559   

results_yearly_smape.csv: 110 rows
   rank                             model  model_type testdata_leakage  \
0     1  TurkForecast-FM-Chronos2-LoRA-v1  fine-tuned               No   
1     2                  timesfm_2_0_500m  pretrained              Yes   
2     3                    LSGT-PFN-torch   zero-shot               No   
3     3             

In [6]:
# Combined rankings: raw metric value + normalized (divided by seasonal_naive, so seasonal_naive = 1.0)
COMBINED_OUTPUTS = {
    'eval_metrics/MASE[0.5]': 'results_yearly_mase_normalized.csv',
    'eval_metrics/sMAPE[0.5]': 'results_yearly_smape_normalized.csv',
    'eval_metrics/mean_weighted_sum_quantile_loss': 'results_yearly_quantile_normalized.csv',
}

BASELINE = 'seasonal_naive'

for metric, out_path in COMBINED_OUTPUTS.items():
    df = yearly_final[['model'] + META_COLUMNS + [metric]].dropna(subset=[metric]).copy()

    baseline_val = df.loc[df['model'] == BASELINE, metric]
    if baseline_val.empty:
        print(f'WARNING: {BASELINE} not found for {metric}, skipping {out_path}')
        continue
    baseline_val = baseline_val.iloc[0]

    df['normalized'] = df[metric] / baseline_val
    df = df.sort_values(metric).reset_index(drop=True)
    df['rank'] = df[metric].rank(method='min').astype(int)
    df = df[['rank', 'model'] + META_COLUMNS + [metric, 'normalized']]
    df.to_csv(out_path, index=False)
    print(f'{out_path}: {len(df)} rows (baseline {BASELINE}={baseline_val:.4f})')
    print(df.head(), '\n')

results_yearly_mase_normalized.csv: 110 rows (baseline seasonal_naive=3.9660)
   rank                             model  model_type testdata_leakage  \
0     1                    tempo_ensemble  fine-tuned              Yes   
1     2  TurkForecast-FM-Chronos2-LoRA-v1  fine-tuned               No   
2     3                  timesfm_2_0_500m  pretrained              Yes   
3     4                    LSGT-PFN-torch   zero-shot               No   
4     4                             GRAIN         NaN              NaN   

   eval_metrics/MASE[0.5]  normalized  
0                2.398107    0.604673  
1                2.535312    0.639269  
2                2.537595    0.639845  
3                2.898559    0.730860  
4                2.898559    0.730860   

results_yearly_smape_normalized.csv: 110 rows (baseline seasonal_naive=0.1635)
   rank                             model  model_type testdata_leakage  \
0     1  TurkForecast-FM-Chronos2-LoRA-v1  fine-tuned               No   
1     2 